In [1]:
import os
import json
import textwrap

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from fm2p.utils.files import read_h5

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

/home/dylan/anaconda3/envs/fm3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SESSION = os.environ.get(
    'FM2P_SESSION',
    '/home/dylan/Storage/freely_moving_data/V1PPC/cohort02_recordings/251016_DMM_DMM061_pos18/fm1/251016_DMM_DMM061_fm_01')
SPIKE_SUFFIX = ''
SAVE_DIR = None

FILES = {'beh_h5': SESSION + '_behavior.h5',
         'beh_json': SESSION + '_behavior_summary.json',
         'neural': SESSION + '_neural{}.h5'.format(SPIKE_SUFFIX),
         'nv_h5': SESSION + '_nonvisual.h5',
         'nv_json': SESSION + '_nonvisual.json'}
for k, p in FILES.items():
    print('{:9s} {}  {}'.format(k, 'ok     ' if os.path.isfile(p) else 'MISSING', p))

In [ ]:
def S_(x):
    if isinstance(x, np.ndarray) and x.ndim == 0:
        x = x.item()
    return x.decode() if isinstance(x, (bytes, np.bytes_)) else x


def L_(x):
    return S_(x).split(',')


B = read_h5(FILES['beh_h5'])
if 'fig' not in B:
    raise RuntimeError('{} has no /fig group -- rerun fm2p.new_analysis on this session.'.format(FILES['beh_h5']))
FG = B['fig']
with open(FILES['beh_json']) as f:
    J = json.load(f)
N = read_h5(FILES['neural']) if os.path.isfile(FILES['neural']) else None
V = read_h5(FILES['nv_h5']) if os.path.isfile(FILES['nv_h5']) else None
VJ = None
if os.path.isfile(FILES['nv_json']):
    with open(FILES['nv_json']) as f:
        VJ = json.load(f)
if V is not None and 'meta' not in V:
    print('WARNING: {} predates the figure data -- rerun fm2p.nonvisual_probe.'.format(FILES['nv_h5']))
print('behavior: {}; neural: {}; nonvisual: {}'.format(S_(FG['meta']['session']), N is not None, V is not None))

In [ ]:
K = FG['const']
FS = float(K['FS'])
DT = 1.0 / FS
BINO = float(K['BINO_HALF_DEG'])
EDGE = float(K['FIELD_EDGE_DEG'])
SH_HEAD, STILL, SH_GAZE = float(K['SHIFTED_HEAD']), float(K['STILL_GAZE']), float(K['SHIFTED_GAZE'])
N_NULL = int(K['N_NULL'])
MOTIF_NAMES = L_(K['MOTIF_NAMES'])
STATE_NAMES = L_(K['STATE_NAMES'])

t = B['t']
light = B['light'].astype(bool)
pose_ok, eye_ok = B['pose_ok'].astype(bool), B['eye_ok'].astype(bool)
P, FL, VEL, KIN = B['pillar'], B['flow'], B['velocity'], B['kin']
lab, lab_k = B['class_gated'], B['class_kmeans']
ev, ev_all = B['events_motif'], B['events_all']
state = B['state']
SES = FG['session']
LAND = FG['landing']
COV = FG['cov']

INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
STATE_COLORS = ('#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948', '#b9b8b2')
MOTIF_COLORS = ('#2a78d6', '#eb6834', '#1baf7a', '#6250d6')
CLASS_COLORS = {'comp': '#b158c3', 'nasal': '#2a78d6', 'temporal': '#eb6834', 'none': '#c9c8c2'}
AREA_COLORS = {'V1': '#2a78d6', 'AM': '#eb6834', 'A': '#1baf7a', 'RL': '#6250d6', 'PM': '#e87ba4',
               'AL': '#eda100', 'LM': '#008300', 'P': '#e34948', 'unassigned': '#b9b8b2'}


def style(ax, grid_y=True):
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'):
        ax.spines[sp].set_color(INK2)
    ax.tick_params(colors=INK2, labelsize=8)
    if grid_y:
        ax.grid(axis='y', color=GRID, lw=0.6)
        ax.set_axisbelow(True)


def pfmt(pv):
    if pv is None or not np.isfinite(pv):
        return 'p n/a'
    return 'p < {:.0e}'.format(1.0 / (N_NULL + 1)) if pv <= 1.0 / (N_NULL + 1) + 1e-12 else 'p = {:.3g}'.format(pv)


def runs(mask):
    m = np.concatenate([[0], np.asarray(mask, int), [0]])
    d = np.diff(m)
    return np.stack([np.flatnonzero(d == 1), np.flatnonzero(d == -1)], axis=1)


def show(fig, name):
    fig.tight_layout()
    if SAVE_DIR:
        os.makedirs(SAVE_DIR, exist_ok=True)
        fig.savefig(os.path.join(SAVE_DIR, name + '.png'), dpi=200, bbox_inches='tight')
    plt.show()


def text_block(rows, width=120):
    for head, body in rows:
        print(head.replace('\n', ' '))
        print(textwrap.indent(textwrap.fill(body, width), '    '))
        print()

In [ ]:
val, ffc, ifc = FG['checks']['val'], FG['checks']['ffc'], FG['checks'].get('ifc', {})
ps, fs_ = FG['pil_sum'], FG['flow_sum']
L, D = LAND['light'], LAND['dark']
lvd = LAND.get('light_vs_dark', {})
g = lambda d, k: float(np.mean(d[k])) if k in d else np.nan
n_valid = np.sum(lab != -1)
rob = [('motif events', LAND['light_motif_events']), ('PETH thresholds', LAND['light_peth_thresholds']),
       ('k-means literal', LAND['light_kmeans_literal'])]
orient_yes = (g(L, 'p_d_abs_2s') < 0.05 and g(L, 'd_abs') < 0 and g(lvd, 'p_block_ecc') < 0.05)
rows = [
    ('Data', '{:.1f} min at 40 Hz; pose valid {:.1f}%, eye tracked {:.1f}%; light {:.1f}% of the session ({} light/dark '
     'blocks).'.format(len(t) * DT / 60, 100 * pose_ok.mean(), 100 * eye_ok.mean(), 100 * light.mean(),
                       int(FG['meta']['n_blocks']))),
    ('Checks', 'Re-rendering {} random frames reproduces the stored reconstruction (mean |luminance error| {:.3f}).  '
     'Analytic flow vs finite-differenced reprojection: r = {:.4f}.  Warping each frame onto the one {:.0f} ms later '
     'leaves {:.0f}% of the image change ({:.0f}% eye held; {:.0f}% vs {:.0f}% where the eye moved > 3 deg).'.format(
         int(val['n']), float(val['mean_abs_err']), float(ffc['r']), g(ifc, 'gap_ms'), 100 * g(ifc, 'ratio'),
         100 * g(ifc, 'ratio_frozen'), 100 * g(ifc, 'ratio_eye_moved'), 100 * g(ifc, 'ratio_frozen_eye_moved'))),
    ('Pillar on retina', 'Within the 120 deg reconstruction on {:.1f}% of eye-valid frames.  Zone (light): binocular '
     '{:.1f}%, right-monocular {:.1f}%, left-monocular {:.1f}%, blind {:.1f}%.  Median width {:.1f} deg at {:.0f} '
     'mm.'.format(100 * ps['in_recon'], *[100 * x for x in ps['zone_light']], ps['width_med'], ps['dist_med'])),
    ('Optic flow', 'Median retinal flow {:.0f} deg/s (rotation {:.0f}, translation {:.0f}).  Compensatory: eye cuts '
     'rotational flow by {:.0f}% ({:.0f} frozen vs {:.0f} free); gaze shifts: {:+.0f}%.'.format(
         fs_['total'], fs_['rot'], fs_['trans'], fs_['comp_stab'], fs_['comp_frozen'], fs_['comp_free'],
         -fs_['shift_stab'])),
    ('Movements', 'Slow-phase eye-on-head gain {:.2f} (light {:.2f}, dark {:.2f}); literal k-means calls {:.0f}% of '
     'frames gaze shifts.  Gaze-velocity split: compensatory {:.1f}%, nasal {:.1f}%, temporal {:.1f}%; {} episodes '
     '({:.2f}/s).  {} motif events: {}.'.format(
         abs(FG['vor']['vor_slope']), abs(FG['vor']['vor_slope_light']), abs(FG['vor']['vor_slope_dark']),
         FG['classes']['k_frac'], *FG['classes']['cls_frac'], len(ev_all['onset']),
         len(ev_all['onset']) / (n_valid * DT), len(ev['onset']),
         ', '.join('{} {:.1f}%'.format(nm, 100 * np.mean(ev['motif'] == k)) for k, nm in enumerate(MOTIF_NAMES)))),
    ('Object-directed gaze (light)', 'n = {} shifts.  Pillar binocular {:.1f}% after vs {:.1f}% before.  Shift-shuffle '
     'null {:.1f}% ({}); occupancy null {:.1f}% ({}).  Toward a monocular pillar {:.1f}% vs {:.1f}% (n={}, {}).  '
     'Eccentricity change {:+.1f} vs {:+.1f} deg ({}).'.format(
         int(L['n']), 100 * g(L, 'bino_post'), 100 * g(L, 'bino_pre'), 100 * g(L, 'shift_null'), pfmt(g(L, 'p_shift')),
         100 * g(L, 'occ_null'), pfmt(g(L, 'p_occ')), 100 * g(L, 'toward'), 100 * g(L, 'toward_null'),
         int(L['n_dir']), pfmt(g(L, 'p_toward')), g(L, 'd_abs'), g(L, 'd_abs_null'), pfmt(g(L, 'p_d_abs_2s')))),
    ('Answer', 'Binocular landing: {}.  Orienting: {}'.format(
        J['landing']['light']['verdict'],
        'YES -- in the light shifts turn toward the pillar and cut its eccentricity; in the dark they do neither '
        '(block-level p = {:.3g})'.format(g(lvd, 'p_block_ecc')) if orient_yes else 'not established')),
    ('Dark control', 'n = {}; binocular after {:.1f}% vs shift-shuffle {:.1f}% ({}), occupancy {:.1f}% ({}).  Toward '
     '{:.1f}% vs {:.1f}% ({}); eccentricity {:+.1f} vs {:+.1f} deg ({}).  Light minus dark, net of own chance: toward '
     '{:+.1f} points ({}; raw {:+.1f}), eccentricity {:+.1f} deg ({}; raw {:+.1f}).  Block level: toward p = {:.3g}, '
     'eccentricity p = {:.3g}.'.format(
         int(D['n']), 100 * g(D, 'bino_post'), 100 * g(D, 'shift_null'), pfmt(g(D, 'p_shift')), 100 * g(D, 'occ_null'),
         pfmt(g(D, 'p_occ')), 100 * g(D, 'toward'), 100 * g(D, 'toward_null'), pfmt(g(D, 'p_toward')), g(D, 'd_abs'),
         g(D, 'd_abs_null'), pfmt(g(D, 'p_d_abs_2s')), 100 * g(lvd, 'd_toward'), pfmt(g(lvd, 'p_toward')),
         100 * g(lvd, 'd_toward_raw'), g(lvd, 'd_ecc'), pfmt(g(lvd, 'p_ecc')), g(lvd, 'd_ecc_raw'),
         g(lvd, 'p_block_toward'), g(lvd, 'p_block_ecc'))),
    ('Robustness', '  |  '.join('{}: n={}, {:.1f}% vs shift {:.1f}% ({}), occ {:.1f}% ({})'.format(
        nm, int(LT['n']), 100 * g(LT, 'bino_post'), 100 * g(LT, 'shift_null'), pfmt(g(LT, 'p_shift')),
        100 * g(LT, 'occ_null'), pfmt(g(LT, 'p_occ'))) for nm, LT in rob)),
    ('States (light)', ', '.join('{} {:.1f}%'.format(nm, COV['light']['frac_time'][k]) for k, nm in enumerate(STATE_NAMES))),
]
text_block(rows)

In [ ]:
az_cyc, az_eye = P['az_cyc'], P['az_eye']
lt = light & np.isfinite(az_cyc)
win = int(90 * FS)
cand = np.flatnonzero(lt[:-win])[::int(10 * FS)]
a0 = cand[int(np.nanargmax([np.nanstd(az_cyc[c:c + win]) * np.mean(lt[c:c + win]) for c in cand]))] if len(cand) else 0
sl = slice(a0, a0 + win)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axhspan(-BINO, BINO, color='#cde2fb', lw=0)
for key, col, lw_, nm in (('az_cyc', '#2a78d6', 1.2, 'cyclopean az'), ('az_eye', '#eb6834', 1.0, 'retinal az (R eye)')):
    y = P[key][sl].copy()
    y[1:][np.abs(np.diff(y)) > 180] = np.nan
    ax.plot(t[sl] - t[a0], y, color=col, lw=lw_, label=nm)
ax.set_ylim(-180, 180)
ax.set_yticks([-180, -90, 0, 90, 180])
ax.set_ylabel('azimuth (deg)')
ax.set_xlabel('time (s)')
ax.set_title('Example 90 s (light); shaded = binocular zone (+-{:.0f} deg)'.format(BINO), loc='left', fontsize=10)
ax.legend(fontsize=8, loc='lower right', ncol=2)
style(ax)
show(fig, '02a_pillar_azimuth_example')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t[sl] - t[a0], P['el_eye'][sl], color='#1baf7a', lw=1.0, label='retinal elevation (deg)')
ax.plot(t[sl] - t[a0], P['width_deg'][sl], color='#6250d6', lw=1.0, label='angular width (deg)')
ax.set_xlabel('time (s)')
ax.set_ylabel('deg')
ax.legend(fontsize=8, loc='lower right', ncol=2)
style(ax)
show(fig, '02b_pillar_elevation_width_example')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
for cond, col, lm, off in (('light', '#2a78d6', light, -0.2), ('dark', '#52514e', ~light, 0.2)):
    z = P['zone'][lm]
    z = z[z >= 0]
    ax.bar(np.arange(4) + off, [np.mean(z == k) * 100 for k in range(4)], width=0.38, color=col, label=cond)
ax.set_xticks(range(4))
ax.set_xticklabels(['binoc', 'mono R', 'mono L', 'blind'])
ax.set_ylabel('% of frames')
ax.set_title('Pillar zone occupancy', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '02c_zone_occupancy')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
bins = np.linspace(-180, 180, 73)
for lm, col, nm in ((light, '#2a78d6', 'light'), (~light, '#52514e', 'dark')):
    ax.hist(az_cyc[lm & np.isfinite(az_cyc)], bins, density=True, histtype='step', color=col, lw=1.5, label=nm)
for x in (-EDGE, -BINO, BINO, EDGE):
    ax.axvline(x, color=INK2, lw=0.6, ls=':')
ax.set_xlabel('cyclopean azimuth (deg, + left)')
ax.set_yticks([])
ax.set_title('Pillar bearing', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax, grid_y=False)
show(fig, '02d_pillar_bearing')

In [ ]:
m = light & np.isfinite(P['az_eye'])
H, _, _ = np.histogram2d(P['az_eye'][m], P['el_eye'][m], bins=[np.arange(-180, 181, 5), np.arange(-90, 91, 5)])
H = H.T / max(H.sum(), 1) * 100
fig, ax = plt.subplots(figsize=(7, 3.8))
im = ax.imshow(np.log10(H + 1e-3), origin='lower', extent=[-180, 180, -90, 90], aspect='auto', cmap='Blues',
               vmin=-3, vmax=np.log10(H.max() + 1e-3))
ax.add_patch(plt.Rectangle((-60, -60), 120, 120, fill=False, ec='#eb6834', lw=1.4))
ax.axvline(0, color=INK2, lw=0.5)
ax.set_xlabel('retinal azimuth (deg, + temporal)')
ax.set_ylabel('retinal elevation (deg)')
ax.set_title('Pillar on the right retina, light (log10 % frames)', loc='left', fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.01)
style(ax, grid_y=False)
show(fig, '02e_pillar_on_retina')

In [ ]:
m = np.isfinite(P['dist_mm']) & np.isfinite(P['width_deg'])
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.hexbin(P['dist_mm'][m] / 10, P['width_deg'][m], gridsize=40, bins='log', cmap='Blues', mincnt=1, extent=(0, 70, 0, 60))
ax.set_xlabel('distance to pillar surface (cm)')
ax.set_ylabel('angular width (deg)')
ax.set_title('Size vs distance', loc='left', fontsize=10)
style(ax, grid_y=False)
show(fig, '02f_size_vs_distance')

In [ ]:
print('Median width {:.1f} deg (IQR {:.1f}-{:.1f})'.format(ps['width_med'], *ps['width_q']))
print('Median height {:.0f} deg'.format(ps['height_med']))
print('Median distance {:.1f} cm'.format(ps['dist_med'] / 10))
print('In 120 deg reconstruction {:.1f}%'.format(100 * ps['in_recon']))
print('In tracked (R) eye field {:.1f}%'.format(100 * ps['in_field']))
print('Re-render check: {} frames, mean |lum err| {:.3f}'.format(int(val['n']), float(val['mean_abs_err'])))
print('Zone model: binocular |az| <= {:.0f} deg, temporal edge {:.0f} deg, conjugate horizontal eyes assumed.'.format(BINO, EDGE))

In [ ]:
FF = FG['flow_field']
gaz, gel = FF['grid_az'], FF['grid_el']


def quiver_panel(key, title, name):
    d = FF[key]
    mf, n = d['mean'], int(d['n'])
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    if np.isfinite(mf).any():
        spd = np.hypot(mf[:, 0], mf[:, 1])
        ax.quiver(gaz, gel, mf[:, 0], mf[:, 1], spd, cmap='Blues', clim=(0, np.nanmax(spd)), angles='xy',
                  scale_units='xy', scale=np.nanmax(spd) / 7 + 1e-9, width=0.006)
        ax.set_title('{}\nn={}, |mean field| {:.0f} deg/s'.format(title, n, np.nanmean(spd)), loc='left', fontsize=9)
    ax.set_xlim(-65, 65)
    ax.set_ylim(-65, 65)
    ax.set_aspect('equal')
    ax.set_xlabel('az (+temporal)')
    ax.set_ylabel('el (deg)')
    style(ax, grid_y=False)
    show(fig, name)


quiver_panel('comp_left', 'compensatory, head turning left', '03a_flow_compensatory')

In [ ]:
quiver_panel('nasal', 'nasal gaze shift', '03b_flow_nasal')

In [ ]:
quiver_panel('temporal', 'temporal gaze shift', '03c_flow_temporal')

In [ ]:
quiver_panel('forward', 'forward run (>10 cm/s, |yaw|<30)', '03d_flow_forward')

In [ ]:
def rot_hist(mask, nm, name):
    mm = mask & np.isfinite(FL['rot']) & np.isfinite(FL['rot_frozen'])
    bins = np.linspace(0, 600, 61)
    fig, ax = plt.subplots(figsize=(4.5, 3.3))
    ax.hist(FL['rot_frozen'][mm], bins, histtype='step', color='#52514e', lw=1.5, label='eye frozen')
    ax.hist(FL['rot'][mm], bins, histtype='step', color='#2a78d6', lw=1.5, label='eye free')
    ax.set_xlabel('rotational flow (deg/s)')
    ax.set_title('{}: median {:.0f} vs {:.0f}'.format(nm, np.median(FL['rot_frozen'][mm]), np.median(FL['rot'][mm])),
                 loc='left', fontsize=9)
    ax.legend(fontsize=8, frameon=False)
    ax.set_yticks([])
    style(ax, grid_y=False)
    show(fig, name)


rot_hist((lab == 0) & (np.abs(VEL['head']) > SH_HEAD), 'compensatory', '03e_rotflow_compensatory')

In [ ]:
rot_hist((lab == 1) | (lab == 2), 'gaze shift', '03f_rotflow_gazeshift')

In [ ]:
sp = KIN['speed']
bins = np.arange(0, 41, 2.5)
fig, ax = plt.subplots(figsize=(4.5, 3.3))
for key, col, nm in (('trans', '#1baf7a', 'translation'), ('rot', '#2a78d6', 'rotation')):
    med = [np.nanmedian(FL[key][(sp >= a) & (sp < b)]) if np.any((sp >= a) & (sp < b)) else np.nan
           for a, b in zip(bins[:-1], bins[1:])]
    ax.plot((bins[:-1] + bins[1:]) / 2, med, color=col, lw=2, marker='o', ms=3, label=nm)
ax.set_xlabel('speed (cm/s)')
ax.set_ylabel('median flow (deg/s)')
ax.set_title('Flow components vs running speed', loc='left', fontsize=9)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '03g_flow_vs_speed')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.3))
if 'warp' in ifc:
    bins = np.linspace(0, 1.5, 31)
    ax.hist(ifc['warp'] / ifc['ident'], bins, histtype='step', color='#2a78d6', lw=1.5,
            label='pose (median {:.2f})'.format(float(ifc['ratio'])))
    ax.hist(ifc['warp_frozen'] / ifc['ident_frozen'], bins, histtype='step', color='#52514e', lw=1.5,
            label='eye held (median {:.2f})'.format(float(ifc['ratio_frozen'])))
    ax.axvline(1, color=INK2, lw=0.6, ls=':')
    ax.legend(fontsize=8, frameon=False)
else:
    ax.text(0.5, 0.5, 'warp check skipped', ha='center', transform=ax.transAxes)
ax.set_xlabel('warp error / no-motion error')
ax.set_yticks([])
ax.set_title('Warp check vs rendered frames\nformula check r = {:.4f}'.format(float(ffc['r'])), loc='left', fontsize=9)
style(ax, grid_y=False)
show(fig, '03h_warp_check')

In [ ]:
win = int(20 * FS)
act = np.convolve(((lab == 1) | (lab == 2)) & light & np.isfinite(FL['total']), np.ones(win), mode='valid')
b0 = int(np.argmax(act))
sl2 = slice(b0, b0 + win)
tt = t[sl2] - t[b0]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(tt, FL['rot_frozen'][sl2], color='#52514e', lw=1, label='rotation, eye frozen')
ax.plot(tt, FL['rot'][sl2], color='#2a78d6', lw=1.2, label='rotation, eye free')
ax.plot(tt, FL['trans'][sl2], color='#1baf7a', lw=1.2, label='translation')
for a, b in runs((lab[sl2] == 1) | (lab[sl2] == 2)):
    ax.axvspan(tt[a], tt[min(b, len(tt) - 1)], color='#fde7c2', lw=0)
ax.set_xlabel('time (s); shaded = gaze shift')
ax.set_ylabel('deg/s')
ax.legend(fontsize=8, frameon=False, ncol=3, loc='upper right')
style(ax)
show(fig, '03i_flow_example')

In [ ]:
def class_scatter(L_lab, title, name):
    rng = np.random.default_rng(0)
    ok = L_lab != -1
    fig, ax = plt.subplots(figsize=(5, 4))
    for code_, cn in ((3, 'none'), (0, 'comp'), (1, 'nasal'), (2, 'temporal')):
        idx = np.flatnonzero(L_lab == code_)
        if not len(idx):
            continue
        idx = rng.choice(idx, min(5000, len(idx)), replace=False)
        ax.scatter(VEL['head'][idx], VEL['eye'][idx], s=2, alpha=0.35, color=CLASS_COLORS[cn],
                   label='{} {:.1f}%'.format('other' if code_ == 3 else cn, 100 * np.mean(L_lab[ok] == code_)))
    ax.plot([-600, 600], [600, -600], color=INK2, lw=0.6, ls='--')
    ax.axhline(0, color=INK2, lw=0.5)
    ax.axvline(0, color=INK2, lw=0.5)
    ax.set_xlim(-600, 600)
    ax.set_ylim(-400, 400)
    ax.set_xlabel('head velocity (deg/s)')
    ax.set_ylabel('eye velocity (deg/s)')
    ax.set_title(title, loc='left', fontsize=10)
    ax.legend(fontsize=7, frameon=False, markerscale=5, loc='upper right')
    style(ax, grid_y=False)
    show(fig, name)


class_scatter(lab_k, 'k-means k=3 (literal)', '04a_classes_kmeans')

In [ ]:
class_scatter(lab, 'gaze-velocity split (used)', '04b_classes_gated')

In [ ]:
bins = np.linspace(-800, 800, 81)
fig, ax = plt.subplots(figsize=(5, 3.6))
for code_, cn in ((0, 'comp'), (1, 'nasal'), (2, 'temporal')):
    ax.hist(VEL['gaze'][lab_k == code_], bins, histtype='step', lw=1.0, ls='--', color=CLASS_COLORS[cn])
    ax.hist(VEL['gaze'][lab == code_], bins, histtype='step', lw=1.6, color=CLASS_COLORS[cn], label=cn)
ax.set_yscale('log')
ax.set_xlabel('gaze velocity (deg/s)')
ax.set_title('Gaze velocity by class', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax, grid_y=False)
show(fig, '04c_gaze_velocity_by_class')

In [ ]:
rows_, cols_ = [0, 1, 2], [0, 1, 2, 3]
lab3 = np.where(lab == -2, 3, lab)
M = np.array([[np.sum((lab_k == r) & (lab3 == c)) for c in cols_] for r in rows_], float)
Mn = M / np.maximum(M.sum(1, keepdims=True), 1) * 100
fig, ax = plt.subplots(figsize=(4.5, 3.6))
ax.imshow(Mn, cmap='Blues', vmin=0, vmax=100, aspect='auto')
for i in range(3):
    for j in range(4):
        ax.text(j, i, '{:.0f}%'.format(Mn[i, j]), ha='center', va='center', fontsize=9,
                color='white' if Mn[i, j] > 55 else INK)
ax.set_xticks(range(4))
ax.set_xticklabels(['comp', 'nasal', 'temp', 'other'])
ax.set_yticks(range(3))
ax.set_yticklabels(['comp', 'nasal', 'temp'])
ax.set_xlabel('gaze-velocity split')
ax.set_ylabel('k-means literal')
ax.set_title('Agreement (row %)', loc='left', fontsize=10)
show(fig, '04d_classifier_agreement')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
for lm, nm, off, col in ((light, 'light', -0.2, '#2a78d6'), (~light, 'dark', 0.2, '#52514e')):
    m = lm & (lab != -1)
    ax.bar(np.arange(3) + off, [100 * np.mean(lab[m] == k) for k in (0, 1, 2)], width=0.38, color=col, label=nm)
ax.set_xticks(range(3))
ax.set_xticklabels(['comp', 'nasal', 'temporal'])
ax.set_ylabel('% of valid frames')
ax.set_title('Class occupancy (used), light vs dark', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '04e_class_occupancy')

In [ ]:
cen = FG['classes']['centres_head_eye']
sk = (lab_k == 1) | (lab_k == 2)
print('k-means centres (head, eye deg/s):')
for nm, (h, e) in zip(('comp', 'nasal', 'temporal'), cen):
    print('  {:9s}{:+7.1f}, {:+7.1f}'.format(nm, h, e))
print('k-means "gaze shift" frames with |gaze| < 60 deg/s: {:.0f}%'.format(100 * np.mean(np.abs(VEL['gaze'][sk]) < 60)))
print('k-means motifs (literal): ' + ', '.join('{} {}'.format(n_, int(c_)) for n_, c_ in zip(MOTIF_NAMES, FG['classes']['k_counts'])))
print('Slow-phase eye-on-head slope: {:.2f} (light {:.2f}, dark {:.2f})'.format(
    FG['vor']['vor_slope'], FG['vor']['vor_slope_light'], FG['vor']['vor_slope_dark']))
print('Savitzky-Golay {} samples, order {}.'.format(int(K['SG_WIN']), int(K['SG_ORD'])))

In [ ]:
MO = FG['motifs']
half = int(MO['trace_half'])
tt_m = np.arange(-half, half + 1) * DT * 1000


def plot_motif(k, j):
    d = (1, -1)[j]
    fig, ax = plt.subplots(figsize=(4, 3))
    for i, (key, col) in enumerate((('head', '#2a78d6'), ('eye', '#eb6834'))):
        mu, sd = d * MO['trace_mean'][k, j, i], MO['trace_sd'][k, j, i]
        ax.fill_between(tt_m, mu - sd, mu + sd, color=col, alpha=0.15, lw=0)
        ax.plot(tt_m, mu, color=col, lw=1.6, label=key)
    ax.axvline(0, color=INK2, lw=0.5)
    ax.axhline(0, color=INK2, lw=0.5)
    ax.set_title('{} {}  n={}'.format(MOTIF_NAMES[k], 'nasal' if d == 1 else 'temporal', int(MO['trace_n'][k, j])),
                 loc='left', fontsize=10)
    ax.set_ylabel('deg/s (nasal +)')
    ax.set_xlabel('ms from onset')
    ax.legend(fontsize=7, frameon=False)
    style(ax, grid_y=False)
    show(fig, '05_motif_{}_{}'.format(MOTIF_NAMES[k], 'nasal' if d == 1 else 'temporal'))


plot_motif(0, 0)

In [ ]:
plot_motif(0, 1)

In [ ]:
plot_motif(1, 0)

In [ ]:
plot_motif(1, 1)

In [ ]:
plot_motif(2, 0)

In [ ]:
plot_motif(2, 1)

In [ ]:
plot_motif(3, 0)

In [ ]:
plot_motif(3, 1)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
for k, nm in enumerate(MOTIF_NAMES):
    lg = ev['lag_ms'][(ev['motif'] == k) & np.isfinite(ev['lag_ms'])]
    if len(lg):
        xs = np.sort(lg)
        ax.step(xs, np.arange(1, len(xs) + 1) / len(xs), color=MOTIF_COLORS[k], lw=1.5, label=nm)
ax.axvline(0, color=INK2, lw=0.5)
ax.set_xlabel('head - eye onset lag (ms; - = head first)')
ax.set_ylabel('cumulative')
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '05i_onset_lag_cdf')

In [ ]:
TR = MO['transitions']
lift = TR['lift']
fig, ax = plt.subplots(figsize=(4.2, 3.8))
with np.errstate(divide='ignore', invalid='ignore'):
    ax.imshow(np.log2(lift), cmap='RdBu_r', vmin=-1.5, vmax=1.5)
for i in range(4):
    for j in range(4):
        star = '*' if np.isfinite(TR['z'][i, j]) and TR['z'][i, j] > 1.96 else ''
        ax.text(j, i, '{:.2f}{}'.format(lift[i, j], star), ha='center', va='center', fontsize=8)
ax.set_xticks(range(4))
ax.set_xticklabels(MOTIF_NAMES)
ax.set_yticks(range(4))
ax.set_yticklabels(MOTIF_NAMES)
ax.set_xlabel('next')
ax.set_ylabel('current')
ax.set_title('Transition lift, {} pairs'.format(int(TR['n_pairs'])), loc='left', fontsize=10)
show(fig, '05j_motif_transitions')

In [ ]:
paper_pct = {'HbE': 16.51, 'HwE': 11.60, 'HD': 21.94, 'ED': 49.94}
print('{:<5}{:>8}{:>9}{:>9}{:>10}{:>10}{:>10}{:>10}'.format('motif', 'events', 'this %', 'paper %', 'eye/head',
                                                           'gaze amp', 'eye amp', 'lag ms'))
for k, nm in enumerate(MOTIF_NAMES):
    m = ev['motif'] == k
    print('{:<5}{:>8d}{:>9.1f}{:>9.1f}{:>10.2f}{:>10.1f}{:>10.1f}{:>10.0f}'.format(
        nm, int(m.sum()), 100 * m.mean(), paper_pct[nm], float(MO['eye_head_ratio'][nm]),
        np.nanmedian(np.abs(ev['gaze_amp'][m])), np.nanmedian(np.abs(ev['eye_amp'][m])), np.nanmedian(ev['lag_ms'][m])))
print('\nCross-direction match distance ratio {:.2f} (Hou et al.: 0.22-0.45).'.format(float(MO['match_ratio'])))

In [ ]:
conds = [('light', LAND['light']), ('dark', LAND['dark']), ('motif ev.', LAND['light_motif_events']),
         ('PETH', LAND['light_peth_thresholds']), ('k-means', LAND['light_kmeans_literal'])]


def ci_err(x):
    return [[100 * (x.mean() - np.percentile(x, 2.5))], [100 * (np.percentile(x, 97.5) - x.mean())]]


fig, ax = plt.subplots(figsize=(5.5, 3.8))
for i, (nm, LL) in enumerate(conds):
    if 'bino_post' not in LL:
        continue
    ax.bar(i - 0.27, 100 * LL['bino_pre'], width=0.18, color='#c9c8c2', label='before' if i == 0 else None)
    ax.bar(i - 0.09, 100 * LL['bino_post'], width=0.18, color='#eb6834', label='after' if i == 0 else None)
    ax.bar(i + 0.09, 100 * LL['shift_null'].mean(), width=0.18, color='#86b6ef',
           label='shift-shuffle' if i == 0 else None, yerr=ci_err(LL['shift_null']), capsize=2)
    ax.bar(i + 0.27, 100 * LL['occ_null'].mean(), width=0.18, color='#52514e',
           label='occupancy' if i == 0 else None, yerr=ci_err(LL['occ_null']), capsize=2)
ax.set_xticks(range(len(conds)))
ax.set_xticklabels([c[0] for c in conds], rotation=30, ha='right')
ax.set_ylabel('% pillar binocular')
ax.set_title('Observed vs nulls (95% CI)', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False, loc='upper right')
style(ax)
show(fig, '06a_landing_vs_nulls')

In [ ]:
def null_hist(key, nm, pkey, name):
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    for cond, col in (('light', '#2a78d6'), ('dark', '#52514e')):
        LL = LAND[cond]
        if 'bino_post' not in LL:
            continue
        ax.hist(100 * LL[key], 40, color=col, alpha=0.45, label='{} null'.format(cond))
        ax.axvline(100 * LL['bino_post'], color=col, lw=2, label='{} obs ({})'.format(cond, pfmt(LL[pkey])))
    ax.set_xlabel('% landing binocular')
    ax.set_yticks([])
    ax.set_title('{} null distribution'.format(nm), loc='left', fontsize=10)
    ax.legend(fontsize=7, frameon=False)
    style(ax, grid_y=False)
    show(fig, name)


null_hist('shift_null', 'shift-shuffle', 'p_shift', '06b_shift_null')

In [ ]:
null_hist('occ_null', 'occupancy', 'p_occ', '06c_occupancy_null')

In [ ]:
lvd = LAND.get('light_vs_dark', {})
fig, ax = plt.subplots(figsize=(4, 3.6))
for i, (cond, col) in enumerate((('light', '#2a78d6'), ('dark', '#52514e'))):
    LL = LAND[cond]
    if 'toward' not in LL:
        continue
    q = np.percentile(LL['toward_null'], [2.5, 50, 97.5]) * 100
    ax.plot([i - 0.15, i - 0.15], [q[0], q[2]], color='#86b6ef', lw=3, solid_capstyle='round')
    ax.plot(i - 0.15, q[1], 'o', color='#86b6ef', ms=5)
    ax.plot(i + 0.15, 100 * LL['toward'], 'D', color=col, ms=7)
    ax.text(i, max(q[2], 100 * LL['toward']) + 2, 'n={}\n{}'.format(int(LL['n_dir']), pfmt(LL['p_toward'])),
            ha='center', fontsize=7, color=INK2)
ax.set_xticks([0, 1])
ax.set_xticklabels(['light', 'dark'])
ax.set_xlim(-0.6, 1.6)
have = [c for c in ('light', 'dark') if 'toward' in LAND[c]]
lo_ = min(100 * min(np.percentile(LAND[c]['toward_null'], 2.5), LAND[c]['toward']) for c in have)
hi_ = max(100 * max(np.percentile(LAND[c]['toward_null'], 97.5), LAND[c]['toward']) for c in have)
ax.set_ylim(lo_ - 3, hi_ + 8)
ax.set_ylabel('% shifts toward pillar side')
ax.set_title('Toward a monocular pillar\n(light-dark net {})'.format(pfmt(g(lvd, 'p_toward'))), loc='left', fontsize=10)
style(ax)
show(fig, '06d_toward_pillar')

In [ ]:
half_e = int(0.6 * FS)
tt_e = np.arange(-half_e, half_e + 1) * DT * 1000
fig, ax = plt.subplots(figsize=(7, 3.6))
for cond, lm, col in (('light', light, '#2a78d6'), ('dark', ~light, '#52514e')):
    on = ev_all['onset'][lm[ev_all['onset']]]
    on = on[(on >= half_e) & (on < len(t) - half_e)]
    seg = np.stack([np.abs(az_cyc[i - half_e:i + half_e + 1]) for i in on])
    mu = np.nanmean(seg, 0)
    se = np.nanstd(seg, 0) / np.sqrt(np.sum(np.isfinite(seg), 0))
    ax.fill_between(tt_e, mu - se, mu + se, color=col, alpha=0.2, lw=0)
    ax.plot(tt_e, mu, color=col, lw=1.8, label='{} (n={})'.format(cond, len(on)))
ax.axvline(0, color=INK2, lw=0.6)
ax.axhline(BINO, color='#eb6834', lw=0.8, ls='--', label='binocular edge')
ax.set_xlabel('ms from gaze-shift onset')
ax.set_ylabel('|pillar cyclopean az| (deg)')
ax.set_title('Pillar eccentricity around gaze shifts', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '06e_eccentricity_around_shifts')

In [ ]:
EV = LAND['events']
keep = EV['keep'].astype(bool) & light[ev_all['onset']]
fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.axhspan(-BINO, BINO, color='#fde7c2', lw=0)
ax.axvspan(-BINO, BINO, color='#eeeeea', lw=0)
ax.scatter(EV['az_pre'][keep], EV['az_post'][keep], s=3, alpha=0.35,
           c=np.where(ev_all['direction'][keep] > 0, '#2a78d6', '#eb6834'))
ax.plot([-180, 180], [-180, 180], color=INK2, lw=0.5)
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xlabel('pillar az before (deg)')
ax.set_ylabel('pillar az after (deg)')
ax.set_title('Light: before vs after', loc='left', fontsize=10)
style(ax, grid_y=False)
show(fig, '06f_before_after')

In [ ]:
SENS = LAND['sens']
hw = sorted(SENS, key=float)
x = [float(h) for h in hw]
fig, ax = plt.subplots(figsize=(4.5, 3.6))
ax.plot(x, [100 * SENS[h]['bino_post'] for h in hw], 'D-', color='#eb6834', label='observed')
ax.plot(x, [100 * SENS[h]['shift_null'].mean() for h in hw], 'o-', color='#86b6ef', label='shift-shuffle')
ax.plot(x, [100 * SENS[h]['occ_null'].mean() for h in hw], 's-', color='#52514e', label='occupancy')
for h, xx in zip(hw, x):
    ax.text(xx, 100 * SENS[h]['bino_post'] + 1.5,
            's' * bool(SENS[h]['p_shift'] < 0.05) + 'o' * bool(SENS[h]['p_occ'] < 0.05), ha='center', fontsize=8, color=INK2)
ax.set_xlabel('binocular half-width (deg)')
ax.set_ylabel('% landing binocular (light)')
ax.set_title('Sensitivity to zone size', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '06g_zone_size_sensitivity')

In [ ]:
def null_bar(res, order, title, name, figsize=(5, 3.6)):
    fig, ax = plt.subplots(figsize=figsize)
    for i, nm in enumerate(order):
        LL = res[nm]
        if 'bino_post' not in LL:
            ax.text(i, 2, 'n={}'.format(int(LL.get('n', 0))), ha='center', fontsize=7, color=INK2)
            continue
        for off, key, col in ((-0.18, 'occ_null', '#b9b8b2'), (0.0, 'shift_null', '#86b6ef')):
            q = np.percentile(LL[key], [2.5, 50, 97.5]) * 100
            ax.plot([i + off, i + off], [q[0], q[2]], color=col, lw=3, solid_capstyle='round')
            ax.plot(i + off, q[1], 'o', color=col, ms=5)
        ax.plot(i + 0.18, 100 * LL['bino_post'], 'D', color='#eb6834', ms=6)
        star = ('s' if LL['p_shift'] < 0.05 else '') + ('o' if LL['p_occ'] < 0.05 else '')
        if star:
            ax.text(i + 0.18, 100 * LL['bino_post'] + 1.5, star, ha='center', fontsize=8, color=INK)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(['{}\nn={}'.format(nm, int(res[nm].get('n', 0))) for nm in order], fontsize=8)
    ax.set_ylabel('% landing binocular')
    ax.set_title(title, loc='left', fontsize=10)
    style(ax)
    show(fig, name)


null_bar(LAND['by_motif'], MOTIF_NAMES, 'By motif', '07a_landing_by_motif')

In [ ]:
null_bar(LAND['by_dist'], L_(LAND['by_dist_order']), 'By distance to pillar at shift start', '07b_landing_by_distance')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.6))
if 'block_toward' in lvd:
    bl = lvd['block_light'].astype(bool)
    for xx, sel, col in ((0, bl, '#2a78d6'), (1, ~bl, '#52514e')):
        yy = 100 * lvd['block_toward_net'][sel]
        ax.scatter(np.full(len(yy), xx) + np.linspace(-0.12, 0.12, len(yy)), yy, s=6 + lvd['block_n'][sel] / 2,
                   color=col, alpha=0.8)
    ax.axhline(0, color=INK2, lw=0.6, ls=':')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['light blocks', 'dark blocks'])
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel('% toward pillar minus block chance')
    ax.set_title('Per block, net of chance\nMann-Whitney p = {:.3g}'.format(float(lvd['p_block_toward'])), loc='left', fontsize=10)
style(ax)
show(fig, '07c_blocks_light_dark')

In [ ]:
null_bar(LAND['by_state'], STATE_NAMES, 'By behavioral state at onset', '07d_landing_by_state', figsize=(11, 3.8))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
Sx = state.astype(float)
Sx[Sx < 0] = np.nan
step = max(1, len(Sx) // 2000)
ax.imshow(Sx[::step][None, :], aspect='auto', cmap=ListedColormap(list(STATE_COLORS)), vmin=-0.5,
          vmax=len(STATE_NAMES) - 0.5, extent=[0, t[-1] / 60, 0, 1], interpolation='nearest')
ax.imshow(light[::step].astype(float)[None, :], aspect='auto', cmap=ListedColormap(['#52514e', '#fde7c2']), vmin=0,
          vmax=1, extent=[0, t[-1] / 60, 1.05, 1.3], interpolation='nearest')
ax.set_ylim(0, 1.3)
ax.set_yticks([0.5, 1.175])
ax.set_yticklabels(['state', 'light'])
ax.set_xlabel('minutes')
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in STATE_COLORS], STATE_NAMES, ncol=9, fontsize=7,
          frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.35))
ax.set_title('Ethogram', loc='left', fontsize=10)
style(ax, grid_y=False)
show(fig, '08a_ethogram')

In [ ]:
hx_s, hy_s = FG['traces']['hx_s'], FG['traces']['hy_s']
arena = SES['arena']
c_ = np.vstack([arena, arena[:1]])
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(c_[:, 0] / 10, c_[:, 1] / 10, color=INK, lw=1)
for k in range(len(STATE_NAMES)):
    idx = np.flatnonzero(state == k)
    if not len(idx):
        continue
    idx = rng.choice(idx, min(2500, len(idx)), replace=False)
    ax.scatter(hx_s[idx] / 10, hy_s[idx] / 10, s=2, color=STATE_COLORS[k], alpha=0.5, label=STATE_NAMES[k])
ax.add_patch(plt.Circle(SES['pillar_xy'] / 10, float(SES['pillar_r']) / 10, color=INK))
ax.set_aspect('equal')
ax.set_xlabel('x (cm)')
ax.set_ylabel('y (cm)')
ax.set_title('Head position by state (all epochs)', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False, markerscale=4, loc='upper left', bbox_to_anchor=(1, 1))
style(ax, grid_y=False)
show(fig, '08b_position_by_state')

In [ ]:
x = np.arange(len(STATE_NAMES))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(x - 0.2, COV['light']['frac_time'], width=0.38, color='#2a78d6', label='light')
ax.bar(x + 0.2, COV['dark']['frac_time'], width=0.38, color='#52514e', label='dark')
ax.set_xticks(x)
ax.set_xticklabels(STATE_NAMES, rotation=30, ha='right')
ax.set_ylabel('% of time')
ax.set_title('Time in state', loc='left', fontsize=10)
ax.legend(fontsize=8, frameon=False)
style(ax)
show(fig, '08c_time_in_state')

In [ ]:
nu, es = KIN['nose_up'], FG['traces']['ear_sep']
bins = np.arange(-40, 46, 5)
mid = (bins[:-1] + bins[1:]) / 2
med = [np.nanmedian(es[(nu >= a) & (nu < b)]) if np.sum((nu >= a) & (nu < b)) > 50 else np.nan
       for a, b in zip(bins[:-1], bins[1:])]
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.plot(mid, med, 'o-', color='#2a78d6', ms=3)
ax.axvline(float(FG['state_params']['rear_pitch_deg']), color='#eb6834', lw=1, ls='--')
ax.set_xlabel('nose-up pitch (deg)')
ax.set_ylabel('ear separation (px)')
ax.set_title('Rear check: head nearer camera', loc='left', fontsize=10)
style(ax)
show(fig, '08d_rear_check')

In [ ]:
data, lbl = [], []
for k, nm in enumerate(STATE_NAMES):
    r = runs(state == k)
    if len(r):
        data.append((r[:, 1] - r[:, 0]) * DT)
        lbl.append(nm)
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.boxplot(data, vert=True, showfliers=False, widths=0.6, medianprops={'color': '#eb6834'},
           boxprops={'color': INK2}, whiskerprops={'color': INK2}, capprops={'color': INK2})
ax.set_yscale('log')
ax.set_xticks(range(1, len(lbl) + 1))
ax.set_xticklabels(lbl, rotation=45, ha='right')
ax.set_ylabel('bout duration (s)')
ax.set_title('Bout durations', loc='left', fontsize=10)
style(ax)
show(fig, '08e_bout_durations')

In [ ]:
METRICS, MLABELS = L_(COV['metrics']), L_(COV['labels'])


def cov_table(cond, name):
    M = np.array([COV[cond][m] for m in METRICS], float)
    sd = np.nanstd(M, 1, keepdims=True)
    with np.errstate(invalid='ignore'):
        Z = (M - np.nanmean(M, 1, keepdims=True)) / np.where(sd > 0, sd, 1)
    fig, ax = plt.subplots(figsize=(7.5, 7))
    ax.imshow(np.nan_to_num(Z), cmap='RdBu_r', vmin=-2.5, vmax=2.5, aspect='auto')
    for i in range(M.shape[0]):
        for k in range(M.shape[1]):
            if np.isfinite(M[i, k]):
                ax.text(k, i, ('{:.2f}' if abs(M[i, k]) < 10 else '{:.0f}').format(M[i, k]), ha='center',
                        va='center', fontsize=7, color=INK)
    ax.set_xticks(range(len(STATE_NAMES)))
    ax.set_xticklabels(STATE_NAMES, rotation=40, ha='right')
    ax.set_yticks(range(len(MLABELS)))
    ax.set_yticklabels(MLABELS, fontsize=8)
    ax.set_title(cond, loc='left', fontsize=11)
    show(fig, name)


cov_table('light', '09a_covariates_light')

In [ ]:
cov_table('dark', '09b_covariates_dark')

In [ ]:
stp = {k: float(v) for k, v in FG['state_params'].items()}
paras = [
    'TIME BASE  Everything on a 25 ms (40 Hz) grid.  Eye camera {:.1f} Hz, IMU ~37 Hz, top-down 7.5 Hz.'.format(
        float(SES['native_fps'])),
    'POSE  From {}, through the renderer\'s rotation chain (tilt_h = -pfh, tilt_v = -phi).'.format(
        os.path.basename(S_(FG['meta']['npz_path']))),
    'PILLAR  Bearing of the pillar axis at eye height.  Zones in the cyclopean frame: binocular |az| <= {:.0f}, '
    'monocular to {:.0f}, blind beyond.  Conjugate horizontal eye movement assumed for the untracked left eye.'.format(BINO, EDGE),
    'OPTIC FLOW  d_dot = -omega x d - (v - (v.d)d)/rho on a 15x15 grid over the 120 deg field; rho by ray casting '
    '(floor, walls of {:.0f} mm, pillar).  Eye-frozen counterfactual = resting pupil.'.format(float(SES['wall_h'])),
    'CLASSIFICATION  Literal Hou et al. k-means reported, not used.  Downstream: gaze shift |gaze| > {:.0f} deg/s, '
    'compensatory |head| > {:.0f} and |gaze| < {:.0f}, then the previous-frame rule.  Motifs: +-100 ms head/eye velocity '
    '+ acceleration, z-score, k-means k=4 per direction, Hungarian matching.  SG {} samples, order {}.'.format(
        SH_GAZE, SH_HEAD, STILL, int(K['SG_WIN']), int(K['SG_ORD'])),
    'LANDING  Pillar cyclopean azimuth 50 ms before vs after the episode.  {} shuffles; p = (1+#null>=obs)/(1+N).  '
    'Breakdown p-values uncorrected.  Light vs dark on net effects, labels permuted within state; block test on '
    'per-block net effects.'.format(N_NULL),
    'STATES  smoothing {smooth_s} s; rear: nose-up > {rear_pitch_deg} deg for >= {rear_min_s} s, speed < {rear_max_speed} '
    'mm/s; investigate: < {pillar_near_mm} mm from pillar, speed < {invest_max_speed} mm/s, pillar within '
    '+-{invest_bearing_deg} deg; approach: speed > {approach_min_speed} mm/s, closing > {approach_closing} mm/s, within '
    '+-{approach_bearing_deg} deg, >= {approach_min_s} s, reaches pillar within {approach_reach_s} s; wall-follow: < '
    '{wall_near_mm} mm from a wall, speed > {wall_min_speed} mm/s, within {wall_angle_deg} deg of parallel; turn: |yaw '
    'rate| > {turn_rate} deg/s for >= {turn_min_s} s; locomote: speed > {loco_speed} mm/s; rest: speed < {rest_speed} '
    'mm/s and |yaw rate| < {rest_rate} deg/s for >= {rest_min_s} s.'.format(**stp),
    'CAVEATS  One session: all p-values within-session.  Right eye only; binocular zone assumes conjugacy.  Rearing '
    'from IMU pitch.  Head position 7.5 Hz.  The pillar is invisible in the dark.',
]
for para in paras:
    print(textwrap.fill(para, 130))
    print()

In [ ]:
GROUPS = L_(N['meta']['groups'])
GROUP_LABELS = {'retinal_pillar': 'retinal pillar', 'retinal_flow': 'retinal flow', 'eye': 'eye pos/vel',
                'head': 'head ang/vel', 'speed': 'speed', 'pupil': 'pupil', 'ego_pillar': 'ego pillar',
                'allo': 'allo pos/hd', 'state': 'state'}
AREA = np.array(L_(N['cells']['area']))
AREAS = sorted(set(AREA))
ENC, RF, OVC = N['encoding'], N['refframe'], N['ovc']
sig = ENC['sig'].astype(bool)
es, eb, eg = RF['eye_shift'], RF['eye_beta'], RF['eye_models']
rel = es['reliable'].astype(bool)
LAYER_ORDER = ['gain_basis', 'retinal_basis', 'v1_recorded', 'v1_gainfit', 'v1_gainfit_noisy']
med_ = lambda x, m: np.nanmedian(x[m]) if m.any() else np.nan

rows = [('Spikes', '{} triangle-smoothed {:.1f} s.  Areas: {}.'.format(
    S_(N['meta']['spike_key']), float(N['meta']['smooth_s']), ', '.join('{} {}'.format(a, int(np.sum(AREA == a))) for a in AREAS))),
    ('Encoding', '{} / {} cells beat both a circular-shift null and the light+drift model; median full R2 {:.4f}.  Cells '
     'with a consistent unique share (>= 4/5 folds): {}.'.format(
         int(sig.sum()), len(sig), np.nanmedian(ENC['r2_full']),
         ', '.join('{} {}'.format(GROUP_LABELS[gn], int(np.sum(sig & (ENC['unique_folds_pos'][:, i] >= 4) & (ENC['unique'][:, i] > 0))))
                   for i, gn in enumerate(GROUPS)))),
    ('Eye vs head', 'Field-shift ratio (1 eye-centred, 0 head-centred) on {} reliable cells: median {:.2f} [IQR {:.2f}-{:.2f}], '
     'eye-position contrast {:.1f} deg.  Beta scan on {} tuned cells: median index {:.2f}; {} prefer eye end, {} head end.  '
     'Models: {} tuned, {} gain-modulated, {} eye-centred-better, {} head-centred-better.'.format(
         int(rel.sum()), med_(es['ratio'], rel), *(np.nanpercentile(es['ratio'][rel], [25, 75]) if rel.any() else [np.nan] * 2),
         float(es['delta']), int(eb['tuned'].sum()), med_(eb['index'], eb['tuned'].astype(bool)),
         int(eb['prefer_first'].sum()), int(eb['prefer_second'].sum()), int(eg['tuned'].sum()), int(eg['gain'].sum()),
         int(eg['eye_better'].sum()), int(eg['head_better'].sum())))]
if 'eye_models_dark' in RF:
    gd, bd = RF['eye_models_dark'], RF['eye_beta_dark']
    rows.append(('Dark control', 'Pillar invisible: {} "tuned", {} gain-modulated, {} eye-centred-better, {} head-centred-better; '
                 'beta scan {} tuned ({} / {} prefer eye / head end).'.format(
                     int(gd['tuned'].sum()), int(gd['gain'].sum()), int(gd['eye_better'].sum()), int(gd['head_better'].sum()),
                     int(bd['tuned'].sum()), int(bd['prefer_first'].sum()), int(bd['prefer_second'].sum()))))
if 'body_shift' in RF:
    bs, bb = RF['body_shift'], RF['body_beta']
    brel = bs['reliable'].astype(bool)
    rows.append(('Head vs body', 'Body axis = direction of travel.  {} locomotion frames; {} reliable cells, median ratio {:.2f} '
                 '(1 head-centred, 0 body-centred); beta scan {} tuned, median index {:.2f}; {} head, {} body.'.format(
                     int(bs['n_frames']), int(brel.sum()), med_(bs['ratio'], brel), int(bb['tuned'].sum()),
                     med_(bb['index'], bb['tuned'].astype(bool)), int(bb['prefer_first'].sum()), int(bb['prefer_second'].sum()))))
for a, d in N.get('decoders', {}).items():
    if 'eta' in d:
        rows.append(('Decoders ({})'.format(a), 'Head-centred: P within {:.3f}, across eye positions {:.3f}, invariance {:.2f}, '
                     'bias/de {:+.2f}.  Retinal: P within {:.3f}, across {:.3f}, invariance {:.2f}, bias/de {:+.2f}.  '
                     'Within p = {:.3g} / {:.3g}.'.format(
                         d['eta']['P_within'], d['eta']['P_cross'], d['eta']['invariance'], d['eta']['bias_index'],
                         d['rho']['P_within'], d['rho']['P_cross'], d['rho']['invariance'], d['rho']['bias_index'],
                         d['eta']['p_within'], d['rho']['p_within'])))
for a, d in N.get('two_layer', {}).items():
    rows.append(('Two-layer ({})'.format(a), 'Readout CV R2 head-specific / head-centred / retinal: ' + '  |  '.join(
        '{} {:.3f} / {:.3f} / {:.3f}'.format(k, d[k]['r2_head_specific'], d[k]['r2_head'], d[k]['r2_retinal'])
        for k in LAYER_ORDER if k in d)))
st_ = (OVC['stab'] > 0.5) & (OVC['p_stab'] < 0.05)
rows.append(('Object vectors', '{} stable allocentric-vector maps; {} also beat EGO and nuisance.  Median light-dark map r '
             '{:.2f}.'.format(int(st_.sum()), int(OVC['ovc'].sum()), med_(OVC['light_dark_r'], st_))))
text_block(rows)

In [ ]:
r2f = ENC['r2_full']
bins = np.linspace(min(-0.02, np.nanpercentile(r2f, 1)), np.nanpercentile(r2f, 99.5), 50)
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.hist(ENC['null_full'], bins, color='#b9b8b2', label='shuffle null (median per cell)')
ax.hist(r2f, bins, histtype='step', color='#2a78d6', lw=1.6, label='observed')
ax.set_xlabel('full-model CV R2')
ax.set_title('{} / {} cells beat null'.format(int(sig.sum()), len(sig)), loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False)
style(ax, grid_y=False)
show(fig, '12a_full_r2')

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, gn in enumerate(GROUPS):
    for j, a in enumerate(AREAS):
        m = sig & (AREA == a)
        yv = ENC['unique'][m, i]
        xv = i + (j - (len(AREAS) - 1) / 2) * 0.25 + rng.uniform(-0.06, 0.06, len(yv))
        ax.scatter(xv, yv, s=6, color=AREA_COLORS.get(a, '#b9b8b2'), alpha=0.6, label=a if i == 0 else None)
        if len(yv):
            ax.plot([xv.mean() - 0.1, xv.mean() + 0.1], [np.median(yv)] * 2, color=INK, lw=1.6)
ax.axhline(0, color=INK2, lw=0.6)
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([GROUP_LABELS[gn] for gn in GROUPS], rotation=25, ha='right')
ax.set_ylabel('unique R2')
ax.set_title('Unique variance', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False, markerscale=2)
style(ax)
show(fig, '12b_unique_variance')

In [ ]:
w = 0.8 / max(len(AREAS), 1)
fig, ax = plt.subplots(figsize=(7, 3.6))
for j, a in enumerate(AREAS):
    m = sig & (AREA == a)
    fr = [100 * np.mean((ENC['unique_folds_pos'][m, i] >= 4) & (ENC['unique'][m, i] > 0)) if m.any() else 0
          for i in range(len(GROUPS))]
    ax.bar(np.arange(len(GROUPS)) + (j - (len(AREAS) - 1) / 2) * w, fr, width=w, color=AREA_COLORS.get(a, '#b9b8b2'), label=a)
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([GROUP_LABELS[gn] for gn in GROUPS], rotation=25, ha='right')
ax.set_ylabel('% of null-beating cells')
ax.set_title('Cells carrying the group uniquely', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False)
style(ax)
show(fig, '12c_unique_fraction')

In [ ]:
med = [np.nanmedian(ENC['alone'][sig, i]) if sig.any() else np.nan for i in range(len(GROUPS))]
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.barh(np.arange(len(GROUPS)), med, color='#86b6ef')
ax.axvline(np.nanmedian(ENC['r2_base'][sig]) if sig.any() else 0, color=INK2, lw=0.8, ls='--')
ax.set_yticks(range(len(GROUPS)))
ax.set_yticklabels([GROUP_LABELS[gn] for gn in GROUPS])
ax.invert_yaxis()
ax.set_xlabel('median CV R2, group + nuisance alone')
ax.set_title('Group alone', loc='left', fontsize=10)
style(ax, grid_y=False)
show(fig, '12d_group_alone')

In [ ]:
def frames_ratio_hist(kind):
    sh = RF[kind + '_shift']
    rl = sh['reliable'].astype(bool)
    r = np.clip(sh['ratio'][rl], -1.5, 2.5)
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    ax.hist(r, np.linspace(-1.5, 2.5, 33), color='#2a78d6')
    for x_ in (0, 1):
        ax.axvline(x_, color=INK2, lw=0.8, ls='--')
    ax.set_xlabel('field-shift ratio')
    ax.set_title('Reliable fields: n={}, median {:.2f}{}'.format(
        int(rl.sum()), np.nanmedian(r) if len(r) else np.nan,
        '  UNDERPOWERED' if sh['n_frames'] < 3000 else ''), loc='left', fontsize=9)
    style(ax)
    show(fig, '{}_ratio_hist'.format(kind))


def frames_ratio_ci(kind):
    sh = RF[kind + '_shift']
    rl = sh['reliable'].astype(bool)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    if rl.any():
        ci = sh['ci'][rl]
        rr = sh['ratio'][rl]
        o = np.argsort(rr)
        ax.errorbar(np.clip(rr[o], -2, 3), np.arange(len(o)),
                    xerr=[np.clip(rr[o] - ci[o, 0], 0, 5), np.clip(ci[o, 1] - rr[o], 0, 5)],
                    fmt='o', ms=3, color='#2a78d6', ecolor='#86b6ef', elinewidth=1)
        for x_ in (0, 1):
            ax.axvline(x_, color=INK2, lw=0.8, ls='--')
        ax.set_xlim(-2, 3)
    ax.set_yticks([])
    ax.set_xlabel('ratio with 95% block-bootstrap CI')
    ax.set_title('Per-cell uncertainty', loc='left', fontsize=9)
    style(ax, grid_y=False)
    show(fig, '{}_ratio_ci'.format(kind))


def frames_beta_hist(kind):
    be = RF[kind + '_beta']
    tun = be['tuned'].astype(bool)
    edge = tun & ((be['beta_best'] <= be['beta'][0]) | (be['beta_best'] >= be['beta'][-1]))
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    ax.hist(be['index'][tun], np.arange(-1.05, 2.1, 0.1), color='#eb6834')
    for x_ in (0, 1):
        ax.axvline(x_, color=INK2, lw=0.8, ls='--')
    ax.set_xlabel('beta-scan index (1 - beta*)')
    ax.set_title('Tuned n={}; {} at grid edge\ncalls: {} first frame, {} second (4/5 folds)'.format(
        int(tun.sum()), int(edge.sum()), int(be['prefer_first'].sum()), int(be['prefer_second'].sum())), loc='left', fontsize=9)
    style(ax)
    show(fig, '{}_beta_hist'.format(kind))


def frames_beta_profiles(kind):
    be = RF[kind + '_beta']
    tun = be['tuned'].astype(bool)
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    if tun.any():
        Rr = be['r2'][:, tun] - np.nanmax(be['r2'][:, tun], 0)
        ax.plot(1 - be['beta'], Rr, color='#b9b8b2', lw=0.6, alpha=0.6)
        ax.plot(1 - be['beta'], np.nanmedian(Rr, 1), color='#eb6834', lw=2)
    ax.set_xlabel('index')
    ax.set_ylabel('R2 - max')
    ax.set_title('CV R2 profiles over frames (orange = median)', loc='left', fontsize=9)
    style(ax)
    show(fig, '{}_beta_profiles'.format(kind))


def frames_example(kind, k):
    sh = RF[kind + '_shift']
    rl = sh['reliable'].astype(bool)
    score = np.where(rl, sh['split_r'], -np.inf)
    ex = [c for c in np.argsort(-score)[:4] if np.isfinite(score[c])]
    xl, ml = (('retinal pillar azimuth (deg)', ('eye low', 'eye high')) if kind == 'eye'
              else ('head-centred pillar azimuth (deg)', ('head-on-body low', 'high')))
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    if k < len(ex):
        c = ex[k]
        xb = np.arange(sh['F_low'].shape[0]) * 10 - 180 + 5
        ax.plot(xb, sh['F_low'][:, c], color='#2a78d6', lw=1.6, label=ml[0])
        ax.plot(xb, sh['F_high'][:, c], color='#eb6834', lw=1.6, label=ml[1])
        ax.set_title('cell {} ({})  ratio {:.2f} [{:.2f}, {:.2f}]'.format(c, AREA[c], sh['ratio'][c], *sh['ci'][c]),
                     loc='left', fontsize=9)
        ax.legend(fontsize=7, frameon=False)
    ax.set_xlabel(xl)
    ax.set_xlim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    style(ax)
    show(fig, '{}_example_{}'.format(kind, k))


frames_ratio_hist('eye')

In [ ]:
frames_ratio_ci('eye')

In [ ]:
frames_beta_hist('eye')

In [ ]:
frames_beta_profiles('eye')

In [ ]:
frames_example('eye', 0)

In [ ]:
frames_example('eye', 1)

In [ ]:
frames_example('eye', 2)

In [ ]:
frames_example('eye', 3)

In [ ]:
frames_ratio_hist('body') if 'body_shift' in RF else print('no head-vs-body analysis in this file')

In [ ]:
frames_ratio_ci('body') if 'body_shift' in RF else None

In [ ]:
frames_beta_hist('body') if 'body_beta' in RF else None

In [ ]:
frames_beta_profiles('body') if 'body_beta' in RF else None

In [ ]:
frames_example('body', 0) if 'body_shift' in RF else None

In [ ]:
frames_example('body', 1) if 'body_shift' in RF else None

In [ ]:
frames_example('body', 2) if 'body_shift' in RF else None

In [ ]:
frames_example('body', 3) if 'body_shift' in RF else None

In [ ]:
r2 = {k[3:]: v for k, v in eg.items() if k.startswith('r2_')}
t_ = eg['tuned'].astype(bool)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(r2['GAIN'][t_] - r2['ADD'][t_], r2['HEAD'][t_] - r2['ADD'][t_], s=10,
           c=[AREA_COLORS.get(a, '#b9b8b2') for a in AREA[t_]])
ax.axhline(0, color=INK2, lw=0.6)
ax.axvline(0, color=INK2, lw=0.6)
ax.set_xlabel('R2 gain-field - additive')
ax.set_ylabel('R2 head-centred - additive')
ax.set_title('Pillar-tuned cells (n={})'.format(int(t_.sum())), loc='left', fontsize=10)
style(ax, grid_y=False)
show(fig, '15a_gain_vs_head')

In [ ]:
keys = (('tuned', 'tuned'), ('gain', 'gain-mod.'), ('eye_better', 'eye-centred >'), ('head_better', 'head-centred >'))
w = 0.8 / max(len(AREAS), 1)
fig, ax = plt.subplots(figsize=(5, 3.6))
for j, a in enumerate(AREAS):
    m = AREA == a
    ax.bar(np.arange(4) + (j - (len(AREAS) - 1) / 2) * w, [int(np.sum(eg[k].astype(bool)[m])) for k, _ in keys],
           width=w, color=AREA_COLORS.get(a, '#b9b8b2'), label=a)
ax.set_xticks(range(4))
ax.set_xticklabels([l for _, l in keys])
ax.set_ylabel('cells')
ax.legend(fontsize=7, frameon=False)
style(ax)
show(fig, '15b_model_counts')

In [ ]:
labels, wv, cv = [], [], []
for a, d in N.get('decoders', {}).items():
    for tg, nm in (('eta', 'head-centred'), ('rho', 'retinal')):
        if tg in d:
            labels.append('{}\n{}'.format(a, nm))
            wv.append(float(d[tg]['P_within']))
            cv.append(float(d[tg]['P_cross']))
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(x - 0.18, wv, width=0.34, color='#2a78d6', label='within eye position')
ax.bar(x + 0.18, cv, width=0.34, color='#eb6834', label='across eye positions')
ax.axhline(0, color=INK2, lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('P = 1 - MAE / chance')
ax.set_title('Population decoders of pillar bearing', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False)
style(ax)
show(fig, '15c_decoders')

In [ ]:
labels, sv, hv, rv, terc = [], [], [], [], []
for a, d in N.get('two_layer', {}).items():
    for k in LAYER_ORDER:
        if k not in d:
            continue
        x_ = d[k]
        labels.append('{}\n{}'.format(a, k.replace('_', ' ')))
        sv.append(x_['r2_head_specific'])
        hv.append(x_['r2_head'])
        rv.append(x_['r2_retinal'])
        terc.append([x_['r2_head_specific_low'], x_['r2_head_specific_mid'], x_['r2_head_specific_high']])
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(x - 0.27, sv, width=0.26, color='#eb6834', label='head-SPECIFIC readout')
ax.bar(x, hv, width=0.26, color='#fbc5a7', label='head-centred readout')
ax.bar(x + 0.27, rv, width=0.26, color='#2a78d6', label='retinal readout')
for i, tv in enumerate(terc):
    ax.plot([i - 0.36, i - 0.27, i - 0.18], tv, 'o-', color=INK, ms=3, lw=0.8,
            label='head-specific at eye low/mid/high' if i == 0 else None)
ax.axhline(0, color=INK2, lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('readout CV R2')
ax.set_title('Two-layer model', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False)
style(ax)
show(fig, '15d_two_layer')

In [ ]:
stable = (OVC['stab'] > 0.5) & (OVC['p_stab'] < 0.05)
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.hist(OVC['stab'][np.isfinite(OVC['stab'])], 40, color='#2a78d6')
ax.axvline(0.5, color=INK2, lw=0.8, ls='--')
ax.set_xlabel('split-half map r (light)')
ax.set_title('{} stable (r > 0.5, p < 0.05)'.format(int(stable.sum())), loc='left', fontsize=10)
style(ax)
show(fig, '16a_ovc_stability')

In [ ]:
d_ = OVC['r2_ALLO'] - OVC['r2_EGO']
d_ = d_[np.isfinite(d_)]
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.hist(np.clip(d_, -0.05, 0.05), np.linspace(-0.05, 0.05, 41), color='#eb6834')
ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlabel('R2 allo-vector - ego-vector')
ax.set_title('{} cells ALLO > EGO (>= 4/5 folds)'.format(int(OVC['allo_beats_ego'].sum())), loc='left', fontsize=10)
style(ax)
show(fig, '16b_allo_vs_ego')

In [ ]:
xd = np.clip(OVC['r2_dark_to_dark'], -0.2, None)
yd = np.clip(OVC['r2_light_to_dark'], -0.2, None)
fig, ax = plt.subplots(figsize=(4.5, 4.2))
ax.scatter(xd, yd, s=6, color='#52514e', alpha=0.5)
ax.scatter(xd[stable], yd[stable], s=14, color='#eb6834', label='stable (clipped at -0.2)')
lim = [-0.2, max(0.05, np.nanmax(np.r_[xd, yd]))]
ax.plot(lim, lim, color=INK2, lw=0.6)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel('ALLO R2, dark-trained (CV)')
ax.set_ylabel('ALLO R2, light-trained, tested in dark')
ax.legend(fontsize=7, frameon=False)
style(ax, grid_y=False)
show(fig, '16c_ovc_dark_transfer')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.hist(OVC['light_dark_r'][stable & np.isfinite(OVC['light_dark_r'])], np.linspace(-1, 1, 21), color='#1baf7a')
ax.set_xlabel('light vs dark map r (stable cells)')
style(ax)
show(fig, '16d_ovc_light_dark_r')

In [ ]:
ex_ovc = [c for c in np.argsort(-np.where(stable, OVC['stab'], -np.inf))[:2] if stable[c]]
na_ = OVC['map_light'].shape[0]
th = np.radians(np.arange(na_ + 1) * 360 / na_ - 180)
rr = np.exp(OVC['dist_edges_log'])


def ovc_map(k, cond):
    if k >= len(ex_ovc):
        print('fewer than {} stable cells'.format(k + 1))
        return
    c = ex_ovc[k]
    Z = OVC['map_' + cond][..., c]
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='polar')
    ax.pcolormesh(th, rr, np.ma.masked_invalid(Z).T, cmap='Blues', shading='flat')
    ax.set_title('cell {} {} (split r {:.2f})'.format(c, cond, OVC['stab'][c]), loc='left', fontsize=9)
    ax.tick_params(labelsize=6)
    show(fig, '16_ovc_cell{}_{}'.format(c, cond))


ovc_map(0, 'light')

In [ ]:
ovc_map(0, 'dark')

In [ ]:
ovc_map(1, 'light')

In [ ]:
ovc_map(1, 'dark')

In [ ]:
VM = V['meta']
KEY_LABEL = {'norm_spikes': 'OMSI', 'oasis_spks': 'OASIS', 'norm_dFF': 'dF/F', 'omsi_asis': 'OMSI as stored'}
KEY_COLOR = {'norm_spikes': '#2a78d6', 'oasis_spks': '#eb6834', 'norm_dFF': '#8a8986', 'omsi_asis': '#9ec1ef'}
COND_COLOR = {'light': '#eda100', 'dark': '#2b2b6b', 'comp': '#b158c3', 'shift': '#2a78d6'}
NV_DT = float(VM['dt'])
SPIKE_KEYS = L_(VM['spike_keys'])
print('{}: eye scale {:.0f} px; slow-phase gain light {:.2f}, dark {:.2f}; {} cells, {} with a sparse-noise RF.\n'.format(
    S_(V['session']), float(V['eye_scale_px']), abs(float(VM['vor_light'])), abs(float(VM['vor_dark'])),
    int(V['n_cells']), int(VM['rf_n_fit'])))
text_block(list(VJ['verdicts'].items()))

In [ ]:
CW = {k: tuple(V['const'][k]) for k in ('BASE_WIN', 'MATCH_WIN', 'SHIFT_WIN')}
print('baseline window {} s, match window {} s, shift window {} s'.format(CW['BASE_WIN'], CW['MATCH_WIN'], CW['SHIFT_WIN']))


def comp_get(cond, key='norm_spikes'):
    return V.get('comp_{}_{}'.format(key, cond))


def comp_balance(cond):
    c = comp_get(cond)
    if c is None:
        print('no {} pairs'.format(cond))
        return
    bal = c['balance']
    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    for nm, col, k in (('gaze shift', COND_COLOR['shift'], 'shift'), ('compensatory', COND_COLOR['comp'], 'comp')):
        ax.scatter(bal['head_peak'][k], bal['head_amp'][k], s=8, c=col, edgecolors='white', linewidths=0.4, label=nm)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('peak head velocity (deg/s)')
    ax.set_ylabel('head amplitude (deg)')
    ax.set_title('{}: {} matched pairs'.format(cond, int(c['n_pairs'])), loc='left', fontsize=10)
    ax.legend(fontsize=7, frameon=False)
    style(ax)
    show(fig, 'NV2_balance_{}'.format(cond))


def comp_gazedisp(cond):
    c = comp_get(cond)
    if c is None:
        return
    bal = c['balance']
    gd = [np.abs(bal['gaze_disp']['shift']), np.abs(bal['gaze_disp']['comp'])]
    bins = np.linspace(0, np.nanpercentile(gd[0], 98), 30)
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    ax.hist(gd[0], bins, color=COND_COLOR['shift'], alpha=0.7, label='gaze shift')
    ax.hist(gd[1], bins, color=COND_COLOR['comp'], alpha=0.7, label='compensatory')
    ax.set_xlabel('|gaze displacement| over the movement (deg)')
    ax.set_ylabel('events')
    dI = (np.nanmedian(c['dI_shift']), np.nanmedian(c['dI_comp'])) if 'dI_shift' in c else (np.nan, np.nan)
    ax.set_title('median gaze disp. {:.1f} vs {:.1f} deg\nRF lum. change {:.3f} vs {:.3f}'.format(
        np.nanmedian(gd[0]), np.nanmedian(gd[1]), *dI), loc='left', fontsize=9)
    ax.legend(fontsize=7, frameon=False)
    style(ax)
    show(fig, 'NV2_gazedisp_{}'.format(cond))


def comp_resp(cond):
    c = comp_get(cond)
    if c is None:
        return
    sel = c['selected'].astype(bool)
    rs, rc = c['resp_shift'], c['resp_comp']
    lim = np.nanpercentile(np.abs(np.r_[rs, rc]), 99)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.scatter(rs[~sel], rc[~sel], s=6, c='#c3c2bd', edgecolors='white', linewidths=0.4, label='not selected')
    ax.scatter(rs[sel], rc[sel], s=12, c=COND_COLOR[cond], edgecolors='white', linewidths=0.4, label='responsive')
    ax.plot([-lim, lim], [-lim, lim], color=INK2, lw=0.8, ls='--')
    ax.axhline(0, color=GRID, lw=0.8)
    ax.axvline(0, color=GRID, lw=0.8)
    ax.set_xlabel('gaze-shift response')
    ax.set_ylabel('compensatory response')
    ax.set_title('pooled (fractional): shift {:+.2f}, comp {:+.2f}\nratio {:.2f} [{:.2f}, {:.2f}]'.format(
        float(c['pooled_shift']), float(c['pooled_comp']), float(c['pooled_ratio']), *c['pooled_ratio_ci']),
        loc='left', fontsize=9)
    ax.legend(fontsize=7, frameon=False)
    style(ax)
    show(fig, 'NV2_responses_{}'.format(cond))


comp_balance('light')

In [ ]:
comp_gazedisp('light')

In [ ]:
comp_resp('light')

In [ ]:
comp_balance('dark')

In [ ]:
comp_gazedisp('dark')

In [ ]:
comp_resp('dark')

In [ ]:
def reg_get(cond, key='norm_spikes'):
    return V.get('reg_{}_{}'.format(key, cond))


def reg_scatter(cond):
    r = reg_get(cond)
    if r is None:
        print('no {} regression'.format(cond))
        return
    sv, sk = r['p_vis'] < 0.05, r['p_kin'] < 0.05
    uk, uv = r['unique_kin'], r['unique_vis']
    fig, ax = plt.subplots(figsize=(4.8, 4))
    for m, col, nm, s in ((np.ones(len(uk), bool), '#c3c2bd', 'neither', 6), (sk & ~sv, '#eb6834', 'kinematic p<0.05', 12),
                          (sv & ~sk, '#2a78d6', 'visual p<0.05', 12), (sv & sk, '#1baf7a', 'both', 12)):
        ax.scatter(uk[m], uv[m], s=s, c=col, edgecolors='white', linewidths=0.4, label=nm)
    ax.axhline(0, color=GRID, lw=0.8)
    ax.axvline(0, color=GRID, lw=0.8)
    ax.set_xlabel('unique R^2, kinematic')
    ax.set_ylabel('unique R^2, visual')
    ax.set_title('{}: {} shifts, {} cells (chance 5% each)'.format(cond, int(r['n_events']), len(r['cells'])), loc='left', fontsize=9)
    ax.legend(fontsize=7, frameon=False)
    style(ax)
    show(fig, 'NV3_unique_{}'.format(cond))


def reg_pop(cond):
    r = reg_get(cond)
    if r is None:
        return
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    for k, (nm, obs, nul, col) in enumerate((('visual', r['pop_vis'], r['pop_vis_null'], '#2a78d6'),
                                             ('kinematic', r['pop_kin'], r['pop_kin_null'], '#eb6834'))):
        obs = float(obs)
        ax.scatter(np.full(len(nul), k) + np.random.default_rng(k).uniform(-0.15, 0.15, len(nul)), nul, s=6, color='#c3c2bd', zorder=2)
        ax.scatter([k], [obs], s=40, color=col, zorder=3, edgecolors='white')
        ax.text(k, obs, '  p={:.3g}'.format((1 + np.sum(nul >= obs)) / (1 + len(nul))), fontsize=7, color=INK2, va='center')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['visual', 'kinematic'])
    ax.set_ylabel('median unique R^2 over cells')
    ax.set_title('{}: population vs permutation null'.format(cond), loc='left', fontsize=9)
    style(ax)
    show(fig, 'NV3_population_{}'.format(cond))


def reg_corr(cond):
    r = reg_get(cond)
    if r is None:
        return
    names = L_(r['cov_names'])
    fig, ax = plt.subplots(figsize=(4.6, 4))
    im = ax.imshow(r['cov_corr'], vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7, rotation=60)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_title('{}: covariate correlation (one cell)'.format(cond), loc='left', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.04)
    show(fig, 'NV3_covcorr_{}'.format(cond))


reg_scatter('light')

In [ ]:
reg_pop('light')

In [ ]:
reg_corr('light')

In [ ]:
reg_scatter('dark')

In [ ]:
reg_pop('dark')

In [ ]:
reg_corr('dark')

In [ ]:
for nm, key in (('primary RF', 'reg_norm_spikes_light'), ('mirrored RF', 'reg_mirror_light'), ('whole field', 'reg_field_light')):
    r = V.get(key)
    if r is not None:
        print('{:12s} visual {:.4f} (null {:.4f})'.format(nm, float(r['pop_vis']), np.mean(r['pop_vis_null'])))

In [ ]:
head_lead = float(V['timing']['head_lead_ms'])
print('Head leads gaze onset by a median {:.0f} ms in head-involved shifts.'.format(head_lead))


def peth(key, name, cs='all'):
    return V.get('peth_{}_{}_{}'.format(key, name, cs))


def peth_panel(name, title):
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    n = None
    for key in SPIKE_KEYS:
        r = peth(key, name)
        if r is None:
            continue
        n = int(r['n']) if key == 'norm_spikes' else n
        tau = 1000 * r['tau']
        ax.fill_between(tau, r['band'][0], r['band'][1], color=KEY_COLOR[key], alpha=0.15, lw=0)
        ax.plot(tau, r['peth'], color=KEY_COLOR[key], lw=1.6,
                label='{} start {:+.0f} ms [{:+.0f},{:+.0f}]'.format(KEY_LABEL[key], 1000 * float(r['t0']), *(1000 * r['t0_ci'])))
    re_ = peth('norm_spikes', name, 'enhanced')
    if re_ is not None:
        ax.plot(1000 * re_['tau'], re_['peth'], color=KEY_COLOR['norm_spikes'], lw=1.0, ls=':', label='OMSI, enhanced cells only')
    ax.axvline(0, color=INK2, lw=0.8)
    ax.axvspan(-1000 * NV_DT, 0, color=GRID, alpha=0.6, lw=0)
    ax.set_xlim(-600, 800)
    ax.set_xlabel('time from onset (ms)')
    ax.set_ylabel('fractional change')
    ax.set_title('{} (n={})'.format(title, n), loc='left', fontsize=10)
    ax.legend(fontsize=6.5, frameon=False, loc='upper left')
    style(ax)
    show(fig, 'NV4_peth_{}'.format(name))


peth_panel('gaze_dark', 'dark shifts, gaze onset')

In [ ]:
peth_panel('gaze_light', 'light shifts, gaze onset')

In [ ]:
peth_panel('head_dark', 'dark shifts, head onset')

In [ ]:
peth_panel('head_light', 'light shifts, head onset')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for cond, col in (('comp_light', COND_COLOR['comp']), ('matched_shift_light', COND_COLOR['shift'])):
    r = peth('norm_spikes', cond)
    if r is None:
        continue
    ax.fill_between(1000 * r['tau'], r['band'][0], r['band'][1], color=col, alpha=0.15, lw=0)
    ax.plot(1000 * r['tau'], r['peth'], color=col, lw=1.6, label='{} (n={})'.format(cond.replace('_', ' '), int(r['n'])))
ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlim(-600, 800)
ax.set_xlabel('time from head onset (ms)')
ax.set_ylabel('fractional change')
ax.set_title('matched pairs, light, OMSI', loc='left', fontsize=10)
ax.legend(fontsize=7, frameon=False)
style(ax)
show(fig, 'NV4_matched_pairs')

In [ ]:
lc = V.get('lag_check')
if lc is not None:
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    ax.bar(lc['lags'], lc['cc_mean'], color='#2a78d6', width=0.6)
    ax.set_xlabel('lag of dF/F rise after OMSI spike (frames)')
    ax.set_ylabel('mean correlation')
    ax.set_title('OMSI frame-assignment check: peak lag {}'.format(int(lc['peak_lag'])), loc='left', fontsize=10)
    style(ax)
    show(fig, 'NV4_omsi_lag')

In [ ]:
DIR = {k: V['dir_norm_spikes_{}'.format(k)] for k in ('shift_light', 'shift_dark', 'visual_light', 'visual_dark')}
DC = V['dircorr_norm_spikes']


def dir_scatter(a, b, la, lb):
    x, y = DIR[a]['dprime'], DIR[b]['dprime']
    sel = (DIR[a]['p'] < 0.05) & (DIR[b]['p'] < 0.05)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.scatter(x, y, s=6, c='#c3c2bd', edgecolors='white', linewidths=0.4, label='all')
    ax.scatter(x[sel], y[sel], s=14, c='#2a78d6', edgecolors='white', linewidths=0.4, label='both p<0.05')
    lim = np.nanpercentile(np.abs(np.r_[x, y]), 99)
    ax.plot([-lim, lim], [-lim, lim], color=INK2, lw=0.8, ls='--')
    ax.axhline(0, color=GRID, lw=0.8)
    ax.axvline(0, color=GRID, lw=0.8)
    ca, cs = DC['{}__{}__all'.format(a, b)], DC['{}__{}__selective'.format(a, b)]
    ax.set_title('rho {:+.2f} [{:+.2f},{:+.2f}] n={}; ceiling {:.2f}\nselective: rho {:+.2f} n={}'.format(
        float(ca['rho']), *ca['ci'], int(ca['n']), float(ca.get('ceiling', np.nan)), float(cs['rho']), int(cs['n'])),
        loc='left', fontsize=9)
    ax.set_xlabel(la + " d'")
    ax.set_ylabel(lb + " d'")
    ax.legend(fontsize=7, frameon=False)
    style(ax)
    show(fig, 'NV5_{}_vs_{}'.format(a, b))


dir_scatter('shift_dark', 'visual_light', 'dark gaze shift (non-visual)', 'visual motion, light')

In [ ]:
dir_scatter('shift_light', 'visual_light', 'light gaze shift', 'visual motion, light')

In [ ]:
dir_scatter('shift_light', 'shift_dark', 'light gaze shift', 'dark gaze shift')

In [ ]:
dir_scatter('visual_light', 'visual_dark', 'visual motion, light', 'same image motion, dark')

In [ ]:
names = ('shift_light', 'shift_dark', 'visual_light', 'visual_dark')
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.bar(range(4), [float(DIR[n]['reliability']) for n in names], color=['#eda100', '#2b2b6b', '#1baf7a', '#8a8986'], width=0.6)
ax.set_xticks(range(4))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=8)
ax.set_ylabel('split-half reliability (S-B)')
ax.axhline(0, color=INK2, lw=0.8)
ax.set_title('Can these indices correlate at all?', loc='left', fontsize=10)
style(ax)
show(fig, 'NV5_reliability')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.bar(range(4), [np.mean(DIR[n]['p'][np.isfinite(DIR[n]['p'])] < 0.05) for n in names], color='#52514e', width=0.6)
ax.axhline(0.05, color='#d03b3b', lw=0.8, ls='--')
ax.set_xticks(range(4))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=8)
ax.set_ylabel('fraction of cells p < 0.05')
ax.set_title('Direction-selective cells', loc='left', fontsize=10)
style(ax)
show(fig, 'NV5_fraction_selective')

In [ ]:
KC = V['const']
paras = [
    'Eye: angles at the measured scale ({:.0f} px); pupil-from-head rebuilt as ang_offset - theta.'.format(float(V['eye_scale_px'])),
    'Gaze shift: |gaze| > {:.0f} deg/s; onset where |gaze| last rose through {:.0f} deg/s.  Head movement: |head| > {:.0f} '
    'deg/s, onset/offset at {:.0f}.  Compensatory: no gaze-shift frame and |gaze| < {:.0f} deg/s.  Isolated: no other onset '
    'in the preceding {:.1f} s ({:.1f} s for timing).'.format(float(KC['SHIFTED_GAZE']), float(KC['GAZE_ON']), float(KC['HEAD_RUN']),
                                                              float(KC['HEAD_ON']), float(KC['STILL_GAZE']), float(KC['ISO_S']),
                                                              float(KC['ISO_TIMING'])),
    'Matching: log peak head velocity, log head amplitude, time since previous movement (cap 2 s); nearest neighbour within '
    'direction x light, caliper {:.1f} SD.  Windows: baseline {}, match {}, shift {} s.'.format(
        float(KC['CALIPER']), tuple(KC['BASE_WIN']), tuple(KC['MATCH_WIN']), tuple(KC['SHIFT_WIN'])),
    'RF: sparse-noise Gaussian fits, monitor {}x{} px, focal {:.0f} px, + {:.0f} deg mapping uncertainty.  {} fitted cells; '
    'population RF az {:+.0f}, el {:+.0f}, sigma {:.0f} deg.'.format(int(KC['MON_SHAPE'][1]), int(KC['MON_SHAPE'][0]),
                                                                    float(KC['MON_F_PX']), float(KC['MAP_SIGMA']), int(VM['rf_n_fit']),
                                                                    float(VM['rf_pop_az']), float(VM['rf_pop_el']), float(VM['rf_pop_sig'])),
    'Image change: surface labels on a {:.0f} deg grid over +-{:.0f} deg, 50 ms before onset and after movement end.'.format(
        float(KC['GRID_STEP']), float(KC['GRID_HALF'])),
    'Visual motion: RF-weighted horizontal optic flow outside gaze shifts ({:.1f} s before to {:.1f} s after excluded); |u| > {:.0f} '
    'deg/s; head-velocity strata {}; >= {} samples per side.'.format(float(KC['EXCL_PRE']), float(KC['EXCL_POST']), float(KC['U0']),
                                                                    list(KC['HEAD_STRATA'][1:-1].astype(int)), int(KC['MIN_STRATUM'])),
    'Statistics: BH q < 0.1 for responsive cells; {} bootstraps; {} permutations (direction), {} circular shifts (visual '
    'direction); regression nulls {} permutations.  Runtime {:.0f} s.'.format(int(KC['N_BOOT']), int(KC['N_PERM']) // 4,
                                                                            int(KC['N_PERM']) // 10, int(KC['N_SHUF']) // 4,
                                                                            float(VM['runtime_s'])),
]
for para in paras:
    print(textwrap.fill(para, 130))
    print()